# Example: AI Sentiment Pipeline Deep Dive

In this example, we open the news pipeline that runs in this build. In parallel with the intraday engine's 30-minute fires, an hourly cron asks Claude with web search for recent headlines on each ticker, scores every headline in $[-1, +1]$ with a second Claude call, and persists the per-ticker aggregates to disk. At every engine fire, the runner reads the latest such file and consults one number per ticker, the news severity, before deciding whether to auto-execute a proposed trade or route it to evening human review. 

We trace the pipeline end to end on captured artifacts, document the on-disk dictionary the cron writes, and calibrate the threshold that controls auto-execute versus human review.

> __Learning Objectives:__
>
> By the end of this example, we will be able to:
> * __Trace one captured headline through the pipeline:__ Open a real `news-*.jld2` artifact written by the hourly cron, pick one high-severity item, and walk its `text`, `claude_score`, and contribution to that ticker's per-ticker `sentiment` and `severity` aggregates. See exactly what the engine reads at the next fire.
> * __Document the artifact schema and the engine's join:__ Walk the on-disk dictionary keys (`source`, `fire_time`, `sentiment`, `severity`, `counts`, `corpus`) the cron persists, and show how the runner wraps per-ticker severity into the snapshot that the compliance gate consumes. Tie each field back to a line of `production_runner.jl`.
> * __Calibrate the severity threshold against the captured corpus:__ Sweep `news_severity_queue_threshold` over all captured fires and tickers, count the auto-execute and queue splits at each candidate value, and read the trade-off between human review burden and auto-execute risk. Compare with the deployed setting.

Let's get started!

___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via [the `Include.jl` file](./Include.jl). This activates the local [Julia](https://julialang.org) environment and loads all dependencies.

In [1]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

  Activating project at `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-4`
    Updating git-repo `https://github.com/varnerlab/alpaca-markets-sdk.git`
    Updating `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-4/Project.toml`
  [c1a5797d] + Alpaca v0.2.0 `https://github.com/varnerlab/alpaca-markets-sdk.git#`
  [336ed68f] + CSV v0.10.16
  [5ae59095] + Colors v0.13.1
  [a93c6f00] + DataFrames v1.8.2
  [587475ba] + Flux v0.16.10
  [cffab07f] + GraphNeuralNetworks v1.1.0
  [86223c79] + Graphs v1.14.0
  [033835bb] + JLD2 v0.6.4
⌃ [682c06a0] + JSON v0.21.4
  [91a5bcdd] + Plots v1.41.6
  [08abe8d2] + PrettyTables v3.3.2
  [10745b16] + Statistics v1.11.1
  [f3b207a7] + StatsPlots v0.15.8
  [a1b2c3d4] + eCornellAIFinance v0.1.0 `../../code`
  [ade2ca70] ~ Dates ⇒ v1.11.0
  [9a3f8284] ~ Random ⇒ v1.11.0
    Updating `~/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-4/Manifest.toml`
  [47edcb42] + ADTypes v1.22.0
  [14f7f29c] + AMD v0.5.

### Constants

In the code block below, we declare the bindings used throughout this example:

* `NEWS_DIR::String`: the directory where the hourly cron writes one `news-YYYY-MM-DD-HH.jld2` per fire.
* `PRODUCTION_S_MAX::Float64`: the production setting of `news_severity_queue_threshold` from `config/production-config.toml` (= 0.7).
* `S_MAX_GRID::Vector{Float64}`: candidate threshold values used in Task 3.
* `FOCUS_FIRE::String`: the captured-fire filename we trace in Task 1, picked for full ticker coverage.

In [2]:
# --- Step 1: Path to the on-disk news artifacts ---
# `_PATH_TO_NEWS` is set in Include.jl to `<project_root>/lectures/session-4/data/news/`.
# Every captured fire is stored there as `news-YYYY-MM-DD-HH.jld2`.
NEWS_DIR = _PATH_TO_NEWS;

# --- Step 2: Production threshold from config/production-config.toml ---
# `s_max` is the only news-related knob in production. A ticker-fire with severity
# strictly greater than this routes the proposed trade to the evening queue.
PRODUCTION_S_MAX = 0.7;

# --- Step 3: Candidate thresholds for the Task 3 sweep ---
# 21-point grid spanning [0, 1] in 0.05 steps. We compute queue volume at each
# candidate threshold so we can read the trade-off curve.
S_MAX_GRID = collect(0.0:0.05:1.0);

# --- Step 4: The fire we trace in Task 1 ---
# One specific captured fire (May 8, 2026, 18:00 UTC). Picked because it has
# full ticker coverage and at least one clearly loud headline.
FOCUS_FIRE = "news-2026-05-08-18.jld2";

The hourly cron has captured a series of fires in `data/news/`. In the code block below, we return: `fires::Vector{NamedTuple}` (one entry per captured fire with `path`, `fire_time`, `source`, `sentiment::Dict{String,Float64}`, `severity::Dict{String,Float64}`, `counts::Dict{String,Int}`, `corpus::MyNewsCorpus`) and `focus::NamedTuple` (the same fields for the FOCUS_FIRE). 

See [the `MyNewsCorpus` struct](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsCorpus) for the field schema.

In [3]:
(; fires, focus) = let
    # --- Step 1: Enumerate captured fire artifacts (filter out budget-*.json and synthetic-corpus*) ---
    # The news directory contains other files (cost-budget logs, synthetic-corpus
    # caches). We accept only filenames matching `news-YYYY-MM-DD-HH.jld2` and
    # sort them chronologically so the captured timeline is in order.
    pattern = r"^news-\d{4}-\d{2}-\d{2}-\d{2}\.jld2$";
    files = filter(f -> occursin(pattern, f), readdir(NEWS_DIR));
    sort!(files);

    # --- Step 2: Load every fire into a typed NamedTuple ---
    # Each on-disk file holds a Dict{String,Any} with six keys: fire_time,
    # source, sentiment, severity, counts, and corpus. We unpack each one into
    # a NamedTuple with the same field names plus the file path, so the loop
    # below can use dot-access (`fr.severity`) instead of key lookups.
    fires = NamedTuple[];
    for f in files
        d = load_results(joinpath(NEWS_DIR, f));
        push!(fires, (
            path = joinpath(NEWS_DIR, f),         # absolute path on disk
            fire_time = d["fire_time"],           # DateTime stamp the cron wrote at
            source = d["source"],                 # synthetic / newsapi / anthropic_web
            sentiment = d["sentiment"],           # Dict{String,Float64}: per-ticker mean of claude_score
            severity = d["severity"],             # Dict{String,Float64}: per-ticker max of |claude_score|
            counts = d["counts"],                 # Dict{String,Int}: per-ticker n_items
            corpus = d["corpus"]));               # MyNewsCorpus with item-level records
    end

    # --- Step 3: Load the focus fire for Task 1 ---
    # Same loader, applied to the one fire we trace in detail below. Stored
    # separately so Task 1 doesn't have to scan the whole `fires` vector.
    fpath = joinpath(NEWS_DIR, FOCUS_FIRE);
    fd = load_results(fpath);
    focus = (
        path = fpath,
        fire_time = fd["fire_time"],
        source = fd["source"],
        sentiment = fd["sentiment"],
        severity = fd["severity"],
        counts = fd["counts"],
        corpus = fd["corpus"]);

    # --- Step 4: Echo the dataset shape so the reader knows what was loaded ---
    # Single-string interpolation keeps each line in one stream record so the
    # rendered notebook doesn't show stream-boundary artifacts.
    first_t = minimum(fr.fire_time for fr in fires);
    last_t  = maximum(fr.fire_time for fr in fires);
    println("Captured fires: $(length(fires)), spanning $(first_t) to $(last_t).")
    println("Focus fire:     $(FOCUS_FIRE), source = $(focus.source), tickers = $(length(focus.sentiment)), items = $(length(focus.corpus.items)).")

    # Returned tuple destructured into the globals `fires` and `focus`.
    (fires = fires, focus = focus)
end;

ArgumentError: ArgumentError: No file exists at given path: /Users/jeffreyvarner/Desktop/julia_work/eCornell-AI-finance-lectures/lectures/session-4/data/news/news-2026-05-08-18.jld2

___
## Task 1: Trace One Captured Headline

In this task, we follow a single news item from its on-disk record through to the per-ticker `sentiment` and `severity` numbers the engine reads at the next fire. Each item in the captured corpus is an instance of [the `MyNewsItem` struct](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) carrying `ticker`, `text`, `claude_score`, and `source`; per-ticker aggregation then produces:

* `sentiment[ticker]` = mean of `claude_score` across items mentioning that ticker,
* `severity[ticker]` = max of `|claude_score|` across the same items,
* `counts[ticker]` = number of items.

Severity is loudness, not direction: a strongly bullish headline (+0.9) and a strongly bearish headline (-0.9) push severity up by the same amount.

> __What should we see?__
>
> The traced item is one of three headlines about the highest-severity ticker in the focus fire. Its `claude_score` is one of the three values aggregated into that ticker's `sentiment` (their mean), and its absolute value is one of three contenders for the ticker's `severity` (their max). When the item we trace happens to carry the largest absolute score, its `|claude_score|` equals `severity[ticker]` exactly.

The cell-bound result is `traced::NamedTuple` with the item, the contributing per-ticker score set, and the aggregated values.

In [4]:
traced = let
    # --- Step 1: Pick the highest-severity ticker in the focus fire ---
    # `focus.severity` is a Dict{String,Float64}; we sort its entries by descending
    # severity (the `-p[2]` flip) and take the top ticker. `items_for_ticker` is
    # the subset of the corpus that mentions this ticker; the trace will land on
    # one of these items.
    pairs = sort(collect(focus.severity); by = p -> -p[2]);
    ticker = first(pairs[1]);
    items_for_ticker = filter(it -> it.ticker == ticker, focus.corpus.items);
    isempty(items_for_ticker) && error("No items for ticker $(ticker) in focus fire.");

    # --- Step 2: Pick the item that drives severity (max |claude_score|) ---
    # Severity is defined as the maximum of |claude_score| across items mentioning
    # the ticker. So the item that drives the ticker's severity is the one whose
    # absolute Claude score is largest. Argmax over the abs-score vector picks it.
    abs_scores = abs.([it.claude_score for it in items_for_ticker]);
    item = items_for_ticker[argmax(abs_scores)];

    # --- Step 3: Render the item's record ---
    # One block per field of the MyNewsItem record so the reader can see exactly
    # what the engine stored from this headline. Single-string interpolation keeps
    # each line in one stream record so the rendered notebook is clean.
    println("Traced item:")
    println("  fire_time      = $(focus.fire_time)")
    println("  ticker         = $(item.ticker)")
    println("  source         = $(item.source)")
    println("  claude_score   = $(round(item.claude_score, digits = 3))")
    println("  text           = $(item.text)")

    # --- Step 4: Show how this item contributes to per-ticker aggregates ---
    # List every item that mentions the same ticker, marking the one we traced.
    # The reader can verify that the traced item's |claude_score| matches the
    # aggregated severity, and that sentiment is the mean across all of them.
    println()
    println("All items for $(ticker) in this fire:")
    for (i, it) in enumerate(items_for_ticker)
        marker = it === item ? "  <- traced item" : "";
        println("  [$(i)]  claude_score = $(round(it.claude_score, digits = 3))$(marker)")
    end
    println()
    println("Aggregated per-ticker values for $(ticker):")
    println("  sentiment[$(ticker)]  = mean(claude_score)   = $(round(focus.sentiment[ticker], digits = 3))")
    println("  severity[$(ticker)]   = max(|claude_score|)  = $(round(focus.severity[ticker], digits = 3))")
    println("  counts[$(ticker)]     = $(focus.counts[ticker])")

    # Returned tuple: the traced item plus enough context to verify the aggregates.
    (item = item, ticker = ticker, all_scores = [it.claude_score for it in items_for_ticker],
        sentiment = focus.sentiment[ticker], severity = focus.severity[ticker])
end;

UndefVarError: UndefVarError: `focus` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___
## Task 2: Document the Schema and the Engine's Join

In this task, we open the on-disk artifact the cron writes and follow it into the runner at the next fire. Each `news-*.jld2` is a `Dict{String,Any}` with six top-level keys, and the runner consumes just two: it lifts `sentiment` and `severity` into per-ticker dicts, then wraps each ticker's severity in a `(news_severity = ...,)` NamedTuple that [the `gate_check(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#gate_check) accepts as one of its inputs.

Four lines of [the `production_runner.jl` script](./scripts/production_runner.jl) carry that flow:

* Lines 253-262 define [the `latest_news_artifact(fire_time, tickers)` function](./scripts/production_runner.jl), which scans `NEWS_DIR` for the most recent `news-*.jld2` at or before `fire_time` and returns its `sentiment` and `severity` dicts (zero per ticker if the file is missing).
* Lines 603-605 build the per-ticker snapshot the gate consumes: `snaps[t] = (news_severity = news.severity[t],)` for each ticker `t`.
* Lines 620-621 call [the `split_intraday_trades(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades), which routes any trade with `news_severity > news_severity_queue_threshold` to the evening queue.
* Lines 766-767 re-read `news.severity` at the close to flag tickers above `flag_severity_threshold` on tomorrow's ticket.

> __What should we see?__
>
> The on-disk dict has six top-level keys. The engine uses two: `sentiment` (recorded into the engine snapshot for the audit log) and `severity` (the gate input). The corpus, item texts, source, and counts are kept on disk for post-hoc analysis but do not enter the live decision path. This is the entire surface area of news in the current build.

The cell below returns `schema::DataFrame` documenting each on-disk key, plus `snaps::Dict{String,NamedTuple}` echoed for one ticker so the runner's wrap is visible.

In [5]:
(; schema, snaps) = let
    # --- Step 1: Document the on-disk artifact keys ---
    # Six top-level keys per news-*.jld2 file. The Role column tells the reader
    # which ones drive the gate decision (severity) and which are kept for
    # post-hoc analysis only (corpus, counts).
    rows = [
        (Key = "fire_time", Type = "DateTime",
            Role = "Stamps the audit log; engine uses Date(fire_time) for tape rotation."),
        (Key = "source", Type = "String",
            Role = "Pipeline mode: synthetic, newsapi, or anthropic_web."),
        (Key = "sentiment", Type = "Dict{String,Float64}",
            Role = "Per-ticker mean of claude_score; recorded in engine_snapshot for audit."),
        (Key = "severity", Type = "Dict{String,Float64}",
            Role = "Per-ticker max of |claude_score|; THE gate input in production."),
        (Key = "counts", Type = "Dict{String,Int}",
            Role = "Per-ticker n_items; kept for post-hoc analysis, not read by the engine."),
        (Key = "corpus", Type = "MyNewsCorpus",
            Role = "Item-level records (text, claude_score, source); kept for analysis only."),
    ];
    schema = DataFrame(rows);
    println("Schema of news-*.jld2 (written by news_scorer.jl, read by latest_news_artifact):")
    pretty_table(schema; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));

    # --- Step 2: Build the per-ticker gate snapshot the way production_runner.jl does ---
    # The runner wraps each ticker's severity in a NamedTuple
    # `(news_severity = ...,)` so the gate can pattern-match on field names.
    # We replicate the same construction here so the trace mirrors production.
    tickers = sort(collect(keys(focus.severity)));
    snaps = Dict{String,NamedTuple}(t => (news_severity = focus.severity[t],) for t in tickers);

    # --- Step 3: Echo one ticker's snapshot so the wrap is visible ---
    # We pick the highest-severity ticker in the focus fire and print the
    # NamedTuple exactly as the runner would hand it to gate_check. Single-string
    # interpolation keeps each line in one stream record.
    by_sev = sort(collect(focus.severity); by = p -> -p[2]);
    top_ticker, top_sev = first(by_sev);
    println()
    println("Per-ticker snapshot the runner builds at production_runner.jl:459:")
    println("  snaps[\"$(top_ticker)\"] = $(snaps[top_ticker])")
    println("  (focus.severity[\"$(top_ticker)\"] = $(round(top_sev, digits = 3)))")

    (schema = schema, snaps = snaps)
end;

Schema of news-*.jld2 (written by news_scorer.jl, read by latest_news_artifact):
 ----------- ---------------------- --------------------------------------------------------------------------
        Key                   Type                                                                       Role 
     String                 String                                                                     String 
 ----------- ---------------------- --------------------------------------------------------------------------
  fire_time               DateTime       Stamps the audit log; engine uses Date(fire_time) for tape rotation.
     source                 String                       Pipeline mode: synthetic, newsapi, or anthropic_web.
  sentiment   Dict{String,Float64}    Per-ticker mean of claude_score; recorded in engine_snapshot for audit.
   severity   Dict{String,Float64}            Per-ticker max of |claude_score|; THE gate input in production.
     counts       Dict{String,Int} 

UndefVarError: UndefVarError: `focus` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___
## Task 3: Calibrate the Severity Threshold Against the Captured Corpus

In this task, we calibrate the knob `news_severity_queue_threshold`, the threshold denoted $s_{\max}$ in the [companion lecture](eCornell-AI-Finance-S4-Lecture-ProductionOps-May-2026.ipynb) (current setting $s_{\max} = 0.7$), against the corpus the cron has accumulated. At every fire and every ticker, this threshold is what decides whether a hypothetical proposed trade in that ticker auto-executes or routes to the evening queue. A lower threshold routes more trades to evening review (the queue grows); a higher threshold lets more trades auto-execute (the queue shrinks). The trade-off has no a-priori right answer; the calibration table makes the operating-point choice explicit.

> __What should we see?__
>
> Sweeping the threshold from 0 to 1 across all captured ticker-fires:
>
> * At $s_{\max} = 0$, every ticker-fire with any non-zero severity routes to the queue: this is the worst case for auto-execute throughput.
> * At $s_{\max} = 1$, no ticker-fire ever queues on news: the gate is effectively off.
> * The current setting of $0.7$ lands at a specific operating point on the curve; the calibration tells us what fraction of trades the build commits to evening review at that setting.

The cell below returns `sweep_table::DataFrame` with one row per candidate threshold (auto and queue counts and percentages). A follow-up cell consumes that table to render the calibration curve.

In [6]:
(; sweep_table, all_severities) = let
    # --- Step 1: Flatten all (fire, ticker) severity readings into a single vector ---
    # The threshold sweep treats every (fire, ticker) pair as one independent
    # routing decision. With N fires and ~K tickers per fire, the population
    # under sweep is N * K "ticker-fires". We flatten the per-fire severity
    # dicts into one long vector so the count operations in Step 2 are simple.
    all_severities = Float64[];
    for fr in fires
        for (_, sev) in fr.severity
            push!(all_severities, sev);
        end
    end
    n_total = length(all_severities);
    avg_tk = round(n_total / length(fires), digits = 1);
    println("Calibrating against $(length(fires)) fires across $(n_total) ticker-fires (avg $(avg_tk) tickers/fire).")

    # --- Step 2: Sweep s_max over the candidate grid ---
    # For each candidate threshold, count how many ticker-fires exceed it
    # (routed to the queue) vs do not (auto-executed). The Production column
    # marks the row at the deployed setting so the reader can locate the
    # current operating point quickly.
    rows = NamedTuple[];
    for s in S_MAX_GRID
        n_queue = count(>(s), all_severities);
        n_auto  = n_total - n_queue;
        push!(rows, (
            S_max = round(s, digits = 2),
            Auto = n_auto,                                          # count auto-executed
            Queued = n_queue,                                       # count routed to evening queue
            Auto_pct = round(100 * n_auto / n_total, digits = 1),
            Queued_pct = round(100 * n_queue / n_total, digits = 1),
            Production = isapprox(s, PRODUCTION_S_MAX) ? "<-" : "", # marker on production row
        ));
    end
    sweep_table = DataFrame(rows);
    println()
    println("Threshold sweep:")
    pretty_table(sweep_table; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));

    (sweep_table = sweep_table, all_severities = all_severities)
end;

UndefVarError: UndefVarError: `fires` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

The same data rendered as a curve makes the trade-off easier to read. The code block below consumes `sweep_table` and `all_severities` from the previous cell and produces a calibration plot via [the `Plots.jl` package](https://docs.juliaplots.org/stable/); the current setting is overlaid as an open red circle so the reader can locate the operating point at a glance. The cell has no escaping value.

In [7]:
let
    # --- Panel 1: Calibration curve ---
    # x-axis is the candidate threshold, y-axis is the percentage of
    # ticker-fires routed to the queue at that threshold. The curve is
    # monotonically non-increasing in s_max. A red open circle marks the
    # production setting so the desk PM can see the operating point against
    # the rest of the curve at a glance.
    n_total = length(all_severities);
    queued_frac = [count(>(s), all_severities) / n_total for s in S_MAX_GRID];
    p = plot(S_MAX_GRID, 100 .* queued_frac;
        xlabel = "news_severity_queue_threshold (s_max)",
        ylabel = "Ticker-fires routed to queue (%)",
        label = "queued % across captured fires",
        lw = 2.5, color = :steelblue, legend = :topright,
        left_margin = 10Plots.mm, bottom_margin = 8Plots.mm,
        right_margin = 4Plots.mm, top_margin = 4Plots.mm);
    prod_idx = findfirst(s -> isapprox(s, PRODUCTION_S_MAX), S_MAX_GRID);
    if prod_idx !== nothing
        scatter!(p, [S_MAX_GRID[prod_idx]], [100 * queued_frac[prod_idx]];
            markershape = :circle, markersize = 8,
            markercolor = :white, markerstrokecolor = :red, markerstrokewidth = 2.5,
            label = "production setting (s_max = $(PRODUCTION_S_MAX))");
    end
    display(p);
end;

UndefVarError: UndefVarError: `all_severities` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___
## Summary

In this example, we opened the news-driven part of the production engine. The hourly cron `news_scorer.jl --mode=anthropic_web` calls Claude with web search per ticker, scores each headline in $[-1, +1]$ via a second Claude call, and writes per-ticker `sentiment`, `severity`, and `counts` to `data/news/news-YYYY-MM-DD-HH.jld2`. The intraday engine reads the latest such file at every fire via `latest_news_artifact`, takes per-ticker severity, and routes loud-news trades to the evening queue.

> __Key Takeaways:__
>
> * __Per-ticker news scoring is real and runs hourly:__ Each fire produces one severity, one sentiment, and one count per ticker, written to JLD2 and read by the next engine fire. Severity is loudness, not direction; a bullish and a bearish headline of equal strength contribute identically to the gate decision.
> * __Severity is the engine's news input:__ The runner reads severity from the latest news artifact and passes it through the compliance gate at every fire. A reading above the threshold routes that ticker's trade to evening review, leaving the SIM and the EMA-crossover regime detector to decide capital allocation.
> * __The threshold is the single tunable knob:__ Sweeping it across the captured corpus traces a calibration curve from full-review at small values to no-review at large values. The deployed setting commits the engine to a specific evening-queue volume that the calibration table makes explicit.

### Disclaimer

This content is for educational purposes only and does not constitute investment advice. The captured news corpus is a small sample drawn from a paper-trading deployment; production use of LLM-scored news flow demands additional validation, monitoring, and operational controls.

___